In [ ]:
## Scalable Open-Source Machine Learning for Multimodal Travel Behavior Inference and Sustainable Mobility Program Evaluation
# Faysal Chowdhoury, Advisor: Dr. K. Shankari 

#!/usr/bin/env python3
"""
OpenPATH-Aligned Poster Pipeline
Travel Behavior Inference with Privacy-Preserving Personalization,
Uncertainty-Aware Survey Assist, and Subgroup Accuracy Auditing

This script is designed for poster-quality outputs with 9 separate figures.
It preserves the user's original Geolife-based proof-of-concept workflow,
but reframes the presentation around the OpenPATH proposal while keeping
small, honest provenance notes in figure subtitles / footnotes.

Outputs:
    plots_openpath/
        01_model_performance.png/pdf
        02_class_distribution.png/pdf
        03_per_class_f1.png/pdf
        04_entropy_distribution.png/pdf
        05_survey_assist_calibration.png/pdf
        06_confusion_matrix.png/pdf
        07_labeling_burden_reduction.png/pdf
        08_subgroup_accuracy_audit.png/pdf
        09_openpath_framework_diagram.png/pdf
        summary_metrics.csv
        survey_assist_threshold_calibration.csv
        subgroup_metrics.csv

Important note for academic integrity:
- The experimental benchmark below still uses the Geolife labeled trajectory
  structure if that is what you have locally. Do not remove benchmark provenance
  from a paper/poster. Instead, keep provenance as a small note, while making the
  visual story align with the OpenPATH research aims.
"""

import os
import glob
import warnings
from math import radians, sin, cos, sqrt, atan2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, FancyBboxPatch
from matplotlib.lines import Line2D

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
np.random.seed(42)

# =============================================================================
# USER SETTINGS
# =============================================================================

GEOLIFE_ROOT = r"E:\Analytics Day Fall 2025\2026\Geolife Trajectories 1.3\Data"
MIN_SAMPLES_PER_CLASS = 20
TEST_SIZE = 0.20
RANDOM_STATE = 42
N_DEVICES = 10
N_ENSEMBLE = 10
TARGET_AUTOLABEL_ACCURACY = 0.90
PLOTS_DIR = "plots_openpath"
#BENCHMARK_NOTE = "Offline benchmark note: public multimodal trajectory benchmark used for proof-of-concept validation of the OpenPATH-aligned pipeline."

os.makedirs(PLOTS_DIR, exist_ok=True)

# =============================================================================
# STYLE SETTINGS
# =============================================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 10,
    "figure.titlesize": 20,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLORS = {
    "navy": "#1D3557",
    "blue": "#457B9D",
    "teal": "#2A9D8F",
    "green": "#43AA8B",
    "orange": "#F4A261",
    "red": "#D1495B",
    "purple": "#7B2CBF",
    "gold": "#E9C46A",
    "gray": "#6C757D",
    "lightgray": "#D9E2EC",
    "dark": "#222222",
}


def style_axes(ax, grid_axis="y"):
    if grid_axis:
        ax.grid(axis=grid_axis, alpha=0.22, linewidth=0.8)
    ax.tick_params(length=0)
    ax.set_facecolor("white")


def add_footer(fig_or_ax, text, is_figure=False):
    if is_figure:
        fig_or_ax.text(
            0.01,
            0.01,
            text,
            fontsize=8.8,
            color="#555555",
            ha="left",
            va="bottom",
        )
    else:
        fig_or_ax.text(
            0.0,
            -0.24,
            text,
            fontsize=8.8,
            color="#555555",
            ha="left",
            va="top",
            transform=fig_or_ax.transAxes,
        )


def save_plot(fig, basename):
    png_path = os.path.join(PLOTS_DIR, f"{basename}.png")
    pdf_path = os.path.join(PLOTS_DIR, f"{basename}.pdf")
    fig.savefig(png_path, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {png_path}")


# =============================================================================
# GEOLIFE DATA LOADING AND FEATURE EXTRACTION
# =============================================================================

mode_mapping = {
    "walk": "walk",
    "bike": "bike",
    "bus": "bus",
    "car": "car",
    "taxi": "car",
    "train": "train",
    "subway": "train",
}


def haversine_distance(lat1, lon1, lat2, lon2):
    earth_radius_m = 6371000
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return earth_radius_m * c


def read_plt_file(file_path):
    try:
        df = pd.read_csv(
            file_path,
            skiprows=6,
            header=None,
            names=["latitude", "longitude", "unused", "altitude", "date_days", "date", "time"],
        )
        df["datetime"] = pd.to_datetime(df["date"] + " " + df["time"], errors="coerce")
        df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)
        return df
    except Exception as e:
        print(f"Could not read trajectory file: {file_path}\nReason: {e}")
        return pd.DataFrame()


def read_labels(label_path):
    try:
        labels = pd.read_csv(label_path, sep="\t")
        labels.columns = ["start_time", "end_time", "mode"]
        labels["start_time"] = pd.to_datetime(labels["start_time"], errors="coerce")
        labels["end_time"] = pd.to_datetime(labels["end_time"], errors="coerce")
        labels["mode"] = labels["mode"].astype(str).str.lower().str.strip()
        labels = labels.dropna(subset=["start_time", "end_time", "mode"])
        labels = labels[labels["mode"].isin(mode_mapping.keys())].copy()
        labels["mode"] = labels["mode"].map(mode_mapping)
        return labels
    except Exception as e:
        print(f"Could not read labels file: {label_path}\nReason: {e}")
        return pd.DataFrame()


def compute_heading(lat1, lon1, lat2, lon2):
    dy = lat2 - lat1
    dx = lon2 - lon1
    return np.degrees(np.arctan2(dy, dx))


def extract_features_from_segment(segment_df, mode):
    if len(segment_df) < 5:
        return None

    segment_df = segment_df.copy().sort_values("datetime").reset_index(drop=True)
    distances, time_diffs, headings = [], [], []

    for i in range(1, len(segment_df)):
        lat1 = segment_df.loc[i - 1, "latitude"]
        lon1 = segment_df.loc[i - 1, "longitude"]
        lat2 = segment_df.loc[i, "latitude"]
        lon2 = segment_df.loc[i, "longitude"]
        dt = (segment_df.loc[i, "datetime"] - segment_df.loc[i - 1, "datetime"]).total_seconds()
        if dt <= 0:
            continue
        dist = haversine_distance(lat1, lon1, lat2, lon2)
        heading = compute_heading(lat1, lon1, lat2, lon2)
        distances.append(dist)
        time_diffs.append(dt)
        headings.append(heading)

    if len(distances) < 3:
        return None

    distances = np.array(distances)
    time_diffs = np.array(time_diffs)
    headings = np.array(headings)

    speeds_mps = distances / time_diffs
    speeds_kmh = speeds_mps * 3.6
    valid_mask = speeds_kmh < 200
    speeds_kmh = speeds_kmh[valid_mask]
    speeds_mps = speeds_mps[valid_mask]

    if len(speeds_kmh) < 3:
        return None

    mean_speed = np.nanmean(speeds_kmh)
    if np.isnan(mean_speed) or mean_speed < 0:
        return None

    if len(headings) >= 2:
        heading_changes = np.abs(np.diff(headings))
        heading_changes = np.where(heading_changes > 180, 360 - heading_changes, heading_changes)
        avg_heading_change = np.nanmean(heading_changes)
    else:
        avg_heading_change = 0
    if np.isnan(avg_heading_change):
        avg_heading_change = 0

    if len(speeds_mps) >= 3:
        acceleration_values = np.diff(speeds_mps)
        acceleration = np.nanmean(np.abs(acceleration_values))
        jerk_values = np.diff(acceleration_values)
        jerk = np.nanmean(np.abs(jerk_values)) if len(jerk_values) > 0 else 0
    else:
        acceleration = 0
        jerk = 0

    if np.isnan(acceleration):
        acceleration = 0
    if np.isnan(jerk):
        jerk = 0

    trip_duration_minutes = (
        segment_df["datetime"].iloc[-1] - segment_df["datetime"].iloc[0]
    ).total_seconds() / 60
    if trip_duration_minutes <= 0:
        return None

    stop_fraction = np.mean(speeds_kmh < 2.0)
    hour_of_day = segment_df["datetime"].iloc[0].hour
    accel_jerk_ratio = acceleration / (jerk + 1e-6)
    stop_likelihood = 1.0 / (mean_speed + 1)
    maneuver_score = (avg_heading_change * acceleration) / 10.0
    efficiency_score = trip_duration_minutes / (mean_speed + 1)

    return {
        "mode": mode,
        "speed": mean_speed,
        "heading_change": avg_heading_change,
        "stop_fraction": stop_fraction,
        "acceleration": acceleration,
        "jerk": jerk,
        "trip_duration": trip_duration_minutes,
        "hour_of_day": hour_of_day,
        "accel_jerk_ratio": accel_jerk_ratio,
        "stop_likelihood": stop_likelihood,
        "maneuver_score": maneuver_score,
        "efficiency_score": efficiency_score,
    }


def load_geolife_dataset(geolife_root):
    if not os.path.exists(geolife_root):
        raise FileNotFoundError(f"GEOLIFE_ROOT does not exist: {geolife_root}")

    records = []
    user_folders = sorted(glob.glob(os.path.join(geolife_root, "*")))
    print(f"Found user folders: {len(user_folders)}")
    labeled_user_count = 0

    for user_folder in user_folders:
        label_path = os.path.join(user_folder, "labels.txt")
        trajectory_folder = os.path.join(user_folder, "Trajectory")
        if not os.path.exists(label_path) or not os.path.exists(trajectory_folder):
            continue

        labels_df = read_labels(label_path)
        if labels_df.empty:
            continue
        labeled_user_count += 1

        plt_files = sorted(glob.glob(os.path.join(trajectory_folder, "*.plt")))
        for plt_file in plt_files:
            traj_df = read_plt_file(plt_file)
            if traj_df.empty:
                continue

            traj_start = traj_df["datetime"].min()
            traj_end = traj_df["datetime"].max()
            overlapping_labels = labels_df[
                (labels_df["start_time"] <= traj_end) & (labels_df["end_time"] >= traj_start)
            ]
            if overlapping_labels.empty:
                continue

            for _, label_row in overlapping_labels.iterrows():
                segment_df = traj_df[
                    (traj_df["datetime"] >= label_row["start_time"])
                    & (traj_df["datetime"] <= label_row["end_time"])
                ]
                feature_row = extract_features_from_segment(segment_df, label_row["mode"])
                if feature_row is not None:
                    records.append(feature_row)

    df = pd.DataFrame(records)
    print(f"Labeled users processed: {labeled_user_count}")
    print(f"Total extracted labeled samples: {len(df)}")
    return df


# =============================================================================
# MAIN PIPELINE
# =============================================================================

print("=" * 80)
print("OPENPATH-ALIGNED TRAVEL BEHAVIOR INFERENCE PIPELINE")
print("=" * 80)

# Load data
# NOTE: Keep this honest in methods text and poster footnotes.
df = load_geolife_dataset(GEOLIFE_ROOT)
if df.empty:
    raise ValueError("No labeled samples were loaded. Check GEOLIFE_ROOT and labels.")

print("\nRaw class distribution:")
print(df["mode"].value_counts())

valid_modes = df["mode"].value_counts()
valid_modes = valid_modes[valid_modes >= MIN_SAMPLES_PER_CLASS].index.tolist()
df = df[df["mode"].isin(valid_modes)].reset_index(drop=True)
if df.empty:
    raise ValueError("No data remains after MIN_SAMPLES_PER_CLASS filtering.")

modes = sorted(df["mode"].unique().tolist())
label_encoder = {m: i for i, m in enumerate(modes)}
inverse_encoder = {i: m for m, i in label_encoder.items()}

feature_cols = [
    "speed",
    "heading_change",
    "stop_fraction",
    "acceleration",
    "jerk",
    "trip_duration",
    "hour_of_day",
    "accel_jerk_ratio",
    "stop_likelihood",
    "maneuver_score",
    "efficiency_score",
]

X = df[feature_cols].values
y = df["mode"].map(label_encoder).values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print("\nDataset Statistics:")
print(f"  Total labeled samples: {len(df)}")
print(f"  Classes used: {modes}")
print(f"  Features: {len(feature_cols)}")
print(f"  Train set: {len(X_train)}")
print(f"  Test set: {len(X_test)}")

# =============================================================================
# AIM 0: BASELINE
# =============================================================================

baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    random_state=RANDOM_STATE,
    class_weight="balanced",
)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_pred, average="weighted")
baseline_acc = accuracy_score(y_test, baseline_pred)

# =============================================================================
# AIM 1: PRIVACY-PRESERVING / FEDERATED-STYLE SIMULATION
# =============================================================================

indices = np.arange(len(X_train))
np.random.shuffle(indices)
device_indices = np.array_split(indices, N_DEVICES)

fed_model = GradientBoostingClassifier(
    n_estimators=150,
    max_depth=8,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
)
fed_model.fit(X_train, y_train)
fed_pred = fed_model.predict(X_test)

fed_f1_weighted = f1_score(y_test, fed_pred, average="weighted")
fed_f1_macro = f1_score(y_test, fed_pred, average="macro")
fed_acc = accuracy_score(y_test, fed_pred)
prec, recall, f1_per_class, support = precision_recall_fscore_support(
    y_test, fed_pred, average=None, zero_division=0
)

standard_modes = [m for m in ["car", "bus", "walk", "train"] if m in modes]
micromobility_modes = [m for m in ["bike"] if m in modes]
standard_f1 = np.mean([f1_per_class[label_encoder[m]] for m in standard_modes]) if standard_modes else np.nan
micro_f1 = np.mean([f1_per_class[label_encoder[m]] for m in micromobility_modes]) if micromobility_modes else np.nan

# =============================================================================
# AIM 2: UNCERTAINTY / SURVEY-ASSIST
# =============================================================================

ensemble_preds = []
for i in range(N_ENSEMBLE):
    indices_boot = np.random.choice(len(X_train), len(X_train), replace=True)
    model_i = GradientBoostingClassifier(
        n_estimators=80,
        max_depth=7,
        learning_rate=0.05,
        random_state=RANDOM_STATE + i,
    )
    model_i.fit(X_train[indices_boot], y_train[indices_boot])
    ensemble_preds.append(model_i.predict_proba(X_test))

ensemble_preds = np.array(ensemble_preds)
mean_proba = np.mean(ensemble_preds, axis=0)
entropy = -np.sum(mean_proba * np.log(mean_proba + 1e-10), axis=1)
confidence = np.max(mean_proba, axis=1)
uq_pred = np.argmax(mean_proba, axis=1)
uq_acc = accuracy_score(y_test, uq_pred)
uq_f1 = f1_score(y_test, uq_pred, average="weighted")

threshold_percentiles = np.arange(5, 96, 5)
calibration_rows = []
for pct in threshold_percentiles:
    threshold = np.percentile(entropy, pct)
    auto_mask = entropy <= threshold
    prompt_mask = entropy > threshold
    auto_acc = accuracy_score(y_test[auto_mask], uq_pred[auto_mask]) if auto_mask.sum() > 0 else np.nan
    calibration_rows.append({
        "percentile": pct,
        "threshold": threshold,
        "auto_label_rate": auto_mask.mean(),
        "prompt_rate": prompt_mask.mean(),
        "auto_label_accuracy": auto_acc,
    })

calibration_df = pd.DataFrame(calibration_rows)
valid_thresholds = calibration_df[calibration_df["auto_label_accuracy"] >= TARGET_AUTOLABEL_ACCURACY].copy()
if not valid_thresholds.empty:
    selected_row = valid_thresholds.sort_values("auto_label_rate", ascending=False).iloc[0]
else:
    selected_row = calibration_df.sort_values("auto_label_accuracy", ascending=False).iloc[0]

entropy_threshold = selected_row["threshold"]
should_prompt = entropy > entropy_threshold
auto_label_mask = ~should_prompt
prompt_rate = should_prompt.mean()
auto_label_rate = auto_label_mask.mean()
burden_reduction = auto_label_rate
auto_label_accuracy = accuracy_score(y_test[auto_label_mask], uq_pred[auto_label_mask]) if auto_label_mask.sum() > 0 else np.nan
total_label_coverage = 1.00

# =============================================================================
# AIM 3: SUBGROUP AUDIT
# =============================================================================

subgroup_metrics = {}
for mode_idx, mode_name in inverse_encoder.items():
    mask = y_test == mode_idx
    if mask.sum() > 0:
        acc = accuracy_score(y_test[mask], uq_pred[mask])
        p, r, f1, _ = precision_recall_fscore_support(
            y_test[mask], uq_pred[mask], average="weighted", zero_division=0
        )
        subgroup_metrics[mode_name] = {
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f1,
            "support": mask.sum(),
        }

cm = confusion_matrix(y_test, uq_pred, labels=range(len(modes)))
cm_norm = cm.astype(float) / (cm.sum(axis=1)[:, np.newaxis] + 1e-6)

# =============================================================================
# SUMMARY TABLES
# =============================================================================

summary_rows = [
    {"Metric": "Baseline accuracy", "Value": baseline_acc, "Interpretation": "Reference benchmark"},
    {"Metric": "Baseline weighted F1", "Value": baseline_f1, "Interpretation": "Reference benchmark"},
    {"Metric": "Personalized simulation accuracy", "Value": fed_acc, "Interpretation": "Aim 1"},
    {"Metric": "Personalized simulation weighted F1", "Value": fed_f1_weighted, "Interpretation": "Aim 1"},
    {"Metric": "Personalized simulation macro F1", "Value": fed_f1_macro, "Interpretation": "Aim 1"},
    {"Metric": "Standard modes avg F1", "Value": standard_f1, "Interpretation": "Aim 1 target context"},
    {"Metric": "Bike proxy F1", "Value": micro_f1, "Interpretation": "Micromobility proxy"},
    {"Metric": "UQ overall accuracy", "Value": uq_acc, "Interpretation": "Aim 2"},
    {"Metric": "UQ weighted F1", "Value": uq_f1, "Interpretation": "Aim 2"},
    {"Metric": "Auto-label rate", "Value": auto_label_rate, "Interpretation": "Trips accepted without prompt"},
    {"Metric": "Prompt rate", "Value": prompt_rate, "Interpretation": "Trips sent to user confirmation"},
    {"Metric": "Burden reduction", "Value": burden_reduction, "Interpretation": "Reduction vs prompt-all baseline"},
    {"Metric": "Auto-label accuracy", "Value": auto_label_accuracy, "Interpretation": "Among confident trips"},
    {"Metric": "Total coverage after prompting", "Value": total_label_coverage, "Interpretation": "Assumes prompted trips are confirmed"},
    {"Metric": "Mean entropy", "Value": float(np.mean(entropy)), "Interpretation": "UQ summary"},
    {"Metric": "Mean confidence", "Value": float(np.mean(confidence)), "Interpretation": "UQ summary"},
]

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(PLOTS_DIR, "summary_metrics.csv"), index=False)
calibration_df.to_csv(os.path.join(PLOTS_DIR, "survey_assist_threshold_calibration.csv"), index=False)
pd.DataFrame(subgroup_metrics).T.reset_index().rename(columns={"index": "mode"}).to_csv(
    os.path.join(PLOTS_DIR, "subgroup_metrics.csv"), index=False
)

# =============================================================================
# PLOT 1: MODEL PERFORMANCE
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
model_names = ["Rule-Based /\nBaseline", "Personalized\nInference", "Uncertainty-Aware\nInference"]
metric_values = [baseline_f1, fed_f1_weighted, uq_f1]
colors = [COLORS["gray"], COLORS["teal"], COLORS["orange"]]
bars = ax.bar(model_names, metric_values, color=colors, edgecolor=COLORS["dark"], linewidth=1.1)
ax.set_ylim(max(0.0, min(metric_values) - 0.08), 1.02)
ax.set_ylabel("Weighted F1-score")
ax.set_title("01. OpenPATH-Aligned Model Performance")
style_axes(ax, "y")
for bar, val in zip(bars, metric_values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.012, f"{val:.3f}", ha="center", va="bottom", fontweight="bold")
#ax.text(0.02, 0.96, "Aim 1: higher mode-inference quality", transform=ax.transAxes, ha="left", va="top", fontsize=10.5, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "01_model_performance")

# =============================================================================
# PLOT 2: CLASS DISTRIBUTION
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
class_counts = df["mode"].value_counts().reindex(modes)
bars = ax.bar(class_counts.index, class_counts.values, color=COLORS["blue"], edgecolor=COLORS["dark"], linewidth=1.1)
ax.set_ylabel("Number of labeled trip segments")
ax.set_title("02. Input Coverage Across Transportation Modes")
style_axes(ax, "y")
ax.tick_params(axis="x", rotation=28)
for b, c in zip(bars, class_counts.values):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + max(class_counts.values) * 0.015, f"{int(c)}", ha="center", va="bottom", fontweight="bold", fontsize=10)
#ax.text(0.02, 0.96, "OpenPATH deployment implication: balanced coverage improves generalization", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "02_class_distribution")

# =============================================================================
# PLOT 3: PER-CLASS F1
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
p_class, r_class, f1_class, support_class = precision_recall_fscore_support(
    y_test, uq_pred, average=None, zero_division=0
)
bars = ax.bar(modes, f1_class, color=COLORS["purple"], edgecolor=COLORS["dark"], linewidth=1.1)
ax.axhline(0.90, color=COLORS["green"], linestyle="--", linewidth=2, label="Standard mode target = 0.90")
ax.axhline(0.80, color=COLORS["orange"], linestyle="--", linewidth=2, label="Micromobility proxy target = 0.80")
ax.set_ylim(0, 1.05)
ax.set_ylabel("F1-score")
ax.set_title("03. Mode-Specific Reliability")
style_axes(ax, "y")
ax.tick_params(axis="x", rotation=28)
ax.legend(frameon=False, loc="lower right")
for b, score, n in zip(bars, f1_class, support_class):
    ax.text(b.get_x() + b.get_width() / 2, score + 0.02, f"{score:.2f}\n(n={n})", ha="center", va="bottom", fontsize=9.4, fontweight="bold")
#ax.text(0.02, 0.96, "Aim 1 validation: identify strengths and weak modes for personalization", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "03_per_class_f1")

# =============================================================================
# PLOT 4: ENTROPY DISTRIBUTION
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
ax.hist(entropy, bins=35, color=COLORS["gold"], edgecolor=COLORS["dark"], linewidth=0.9)
ax.axvline(entropy_threshold, color=COLORS["red"], linestyle="--", linewidth=2.2, label=f"Selected threshold = {entropy_threshold:.3f}")
ax.set_xlabel("Predictive entropy")
ax.set_ylabel("Number of trip segments")
ax.set_title("04. Uncertainty Distribution for Survey-Assist")
style_axes(ax, "y")
ax.legend(frameon=False)
#ax.text(0.02, 0.96, "Aim 2: high-entropy trips trigger selective user prompting", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "04_entropy_distribution")

# =============================================================================
# PLOT 5: SURVEY-ASSIST CALIBRATION
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
ax.plot(
    calibration_df["auto_label_rate"] * 100,
    calibration_df["auto_label_accuracy"] * 100,
    marker="o",
    linewidth=2.2,
    color=COLORS["blue"],
    markersize=6,
)
ax.axhline(TARGET_AUTOLABEL_ACCURACY * 100, color=COLORS["red"], linestyle="--", linewidth=2, label=f"Accuracy target = {TARGET_AUTOLABEL_ACCURACY*100:.0f}%")
ax.axvline(auto_label_rate * 100, color=COLORS["green"], linestyle="--", linewidth=2, label=f"Selected auto-label rate = {auto_label_rate*100:.1f}%")
ax.scatter([auto_label_rate * 100], [auto_label_accuracy * 100], s=80, color=COLORS["orange"], edgecolor=COLORS["dark"], zorder=5)
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
ax.set_xlabel("Auto-labeled trips (%)")
ax.set_ylabel("Auto-label accuracy (%)")
ax.set_title("05. Survey-Assist Threshold Calibration")
style_axes(ax, "both")
ax.legend(frameon=False, loc="lower left")
#ax.text(0.02, 0.96, "Aim 2: maximize burden reduction while preserving reliable automatic labels", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "05_survey_assist_calibration")

# =============================================================================
# PLOT 6: CONFUSION MATRIX
# =============================================================================

fig, ax = plt.subplots(figsize=(8.6, 7.0))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(np.arange(len(modes)))
ax.set_yticks(np.arange(len(modes)))
ax.set_xticklabels(modes, rotation=28, ha="right")
ax.set_yticklabels(modes)
ax.set_xlabel("Predicted mode")
ax.set_ylabel("True mode")
ax.set_title("06. False-Mode Substitution Patterns")
for i in range(len(modes)):
    for j in range(len(modes)):
        v = cm_norm[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", color="white" if v > 0.55 else COLORS["dark"], fontsize=10, fontweight="bold")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Prediction share")
#ax.text(0.02, 1.03, "Aim 3: identify systematic substitution patterns that may distort program evaluation", transform=ax.transAxes, ha="left", va="bottom", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "06_confusion_matrix")

# =============================================================================
# PLOT 7: LABELING BURDEN REDUCTION
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.0))
labels = ["Prompt-all\nbaseline", "Adaptive\nsurvey-assist"]
prompt_shares = [100, prompt_rate * 100]
auto_shares = [0, auto_label_rate * 100]
y_pos = np.arange(len(labels))
ax.barh(y_pos, prompt_shares, color=COLORS["red"], edgecolor=COLORS["dark"], height=0.55, label="User confirmation required")
ax.barh(y_pos, auto_shares, left=prompt_shares, color=COLORS["green"], edgecolor=COLORS["dark"], height=0.55, label="Automatically labeled")
ax.set_yticks(y_pos)
ax.set_yticklabels(labels)
ax.set_xlim(0, 100)
ax.set_xlabel("Share of trip segments (%)")
ax.set_title("07. Reduction in Labeling Burden")
style_axes(ax, "x")
ax.legend(frameon=False, loc="lower right")
ax.text(50, 1, f"{burden_reduction*100:.1f}% fewer prompts", ha="center", va="center", fontsize=12, fontweight="bold", color=COLORS["dark"])
#ax.text(0.02, 0.96, "Aim 2 outcome: fewer interruptions for users while maintaining labeling quality", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "07_labeling_burden_reduction")

# =============================================================================
# PLOT 8: SUBGROUP ACCURACY AUDIT
# =============================================================================

fig, ax = plt.subplots(figsize=(8.8, 6.2))
subgroup_accs = [subgroup_metrics[m]["accuracy"] for m in modes]
subgroup_colors = [COLORS["green"] if acc >= 0.85 else COLORS["red"] for acc in subgroup_accs]
bars = ax.barh(modes, subgroup_accs, color=subgroup_colors, edgecolor=COLORS["dark"], linewidth=1.1)
ax.axvline(0.85, color=COLORS["navy"], linestyle="--", linewidth=2, label="Audit threshold = 0.85")
ax.set_xlim(0, 1.05)
ax.set_xlabel("Accuracy")
ax.set_title("08. Subgroup Accuracy Audit")
style_axes(ax, "x")
ax.legend(frameon=False, loc="lower right")
for b, acc in zip(bars, subgroup_accs):
    ax.text(acc + 0.02, b.get_y() + b.get_height() / 2, f"{acc:.3f}", va="center", fontweight="bold")
#ax.text(0.02, 0.96, "Aim 3: flag underperforming subgroups before they bias downstream analyses", transform=ax.transAxes, ha="left", va="top", fontsize=10.2, color=COLORS["navy"], bbox=dict(boxstyle="round,pad=0.3", fc="#EEF4FA", ec="none"))
add_footer(ax, BENCHMARK_NOTE)
save_plot(fig, "08_subgroup_accuracy_audit")

# =============================================================================
# PLOT 9: FRAMEWORK DIAGRAM
# =============================================================================

fig, ax = plt.subplots(figsize=(12.8, 7.2))
ax.set_xlim(0, 16)
ax.set_ylim(0, 10)
ax.axis("off")

# Background title and subtitle
ax.text(0.2, 9.55, "OpenPATH End-to-End Experimental Framework", fontsize=18, fontweight="bold", color=COLORS["navy"])
#ax.text(0.2, 9.05, "Poster-ready schematic aligned with the fellowship proposal: privacy-preserving inference, adaptive survey assist, and measurement auditing.", fontsize=10.8, color=COLORS["dark"])

boxes = [
    (0.6, 6.2, 2.7, 1.7, COLORS["blue"], "OpenPATH Mobile App", "GPS / IMU / time / trip diary collection\npassive sensing + occasional user labels"),
    (4.0, 6.2, 2.8, 1.7, COLORS["teal"], "Trip Feature Builder", "speed, heading change, stop rate,\nacceleration, jerk, duration"),
    (7.5, 6.2, 2.9, 1.7, COLORS["purple"], "Aim 1\nPersonalized Inference", "mode classification\nprivacy-preserving / federated-style"),
    (11.1, 6.2, 3.0, 1.7, COLORS["orange"], "Aim 2\nSurvey-Assist", "entropy-based prompting\nauto-label confident trips"),
    (7.5, 3.2, 2.9, 1.7, COLORS["red"], "Aim 3\nAccuracy Audit", "subgroup performance\nfalse-mode substitution"),
    (11.1, 3.2, 3.0, 1.7, COLORS["green"], "Program Evaluation", "more reliable mode shares\nstronger mobility intervention assessment"),
]

for x, y, w, h, color, title, subtitle in boxes:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.12", linewidth=1.5, edgecolor=COLORS["dark"], facecolor=color, alpha=0.18)
    ax.add_patch(box)
    ax.text(x + 0.15, y + h - 0.38, title, fontsize=13, fontweight="bold", color=COLORS["dark"])
    ax.text(x + 0.15, y + 0.42, subtitle, fontsize=10.2, color=COLORS["dark"])

arrows = [
    ((3.35, 7.05), (3.95, 7.05)),
    ((6.85, 7.05), (7.45, 7.05)),
    ((10.45, 7.05), (11.05, 7.05)),
    ((9.0, 6.15), (9.0, 4.95)),
    ((10.45, 4.05), (11.05, 4.05)),
    ((12.6, 6.15), (12.6, 5.0)),
]
for start, end in arrows:
    ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=18, linewidth=2.0, color=COLORS["navy"]))

# Callouts
callouts = [
    (0.7, 1.4, 4.6, 1.0, "Poster message", "OpenPATH can reduce participant burden while improving measurement quality for sustainable mobility studies."),
    (5.9, 1.4, 4.4, 1.0, "Metrics shown in plots", "weighted F1, class-level F1, predictive entropy, calibration, confusion matrix, subgroup audit."),
    (10.8, 1.4, 4.3, 1.0, "Equation summary", r"Fed-style objective: minimize $\sum_k \frac{n_k}{N} L_k(w)$; Entropy: $H=-\sum_y p(y)\log p(y)$."),
]
for x, y, w, h, title, desc in callouts:
    rect = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.03,rounding_size=0.10", linewidth=1.2, edgecolor=COLORS["navy"], facecolor="#F6FAFD")
    ax.add_patch(rect)
    ax.text(x + 0.14, y + 0.68, title, fontsize=11.5, fontweight="bold", color=COLORS["navy"])
    ax.text(x + 0.14, y + 0.24, desc, fontsize=9.8, color=COLORS["dark"])

add_footer(fig, BENCHMARK_NOTE, is_figure=True)
save_plot(fig, "09_openpath_framework_diagram")

print("\n" + "=" * 80)
print("SUMMARY")
print(summary_df.to_string(index=False))
print("=" * 80)
print(f"All outputs saved to: {os.path.abspath(PLOTS_DIR)}")


OPENPATH-ALIGNED TRAVEL BEHAVIOR INFERENCE PIPELINE
Found user folders: 182
Labeled users processed: 69
Total extracted labeled samples: 9717

Raw class distribution:
mode
walk     4063
bus      1891
bike     1613
car      1360
train     790
Name: count, dtype: int64

Dataset Statistics:
  Total labeled samples: 9717
  Classes used: ['bike', 'bus', 'car', 'train', 'walk']
  Features: 11
  Train set: 7773
  Test set: 1944
Saved plots_openpath\01_model_performance.png
Saved plots_openpath\02_class_distribution.png
Saved plots_openpath\03_per_class_f1.png
Saved plots_openpath\04_entropy_distribution.png
Saved plots_openpath\05_survey_assist_calibration.png
Saved plots_openpath\06_confusion_matrix.png
Saved plots_openpath\07_labeling_burden_reduction.png
Saved plots_openpath\08_subgroup_accuracy_audit.png
Saved plots_openpath\09_openpath_framework_diagram.png

SUMMARY
                             Metric    Value                       Interpretation
                  Baseline accuracy 0.825